In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

## 1. Abrir um CSV com `;` e vírgula decimal

In [ ]:
df = pd.read_csv("Tabela4.csv", sep=";", decimal=",", encoding="latin-1", skiprows=1)
df.head()

## 2. Limpar colunas vazias

In [ ]:
df = df.dropna(axis=1, how="all") 
df = df.dropna(how="all")           
df.columns = [str(c).strip() for c in df.columns]  

print(df.shape)
df.info()

## 3. Estados ordenados pelo maior IDH em 2024

In [ ]:
rank_2024 = df.sort_values("2024", ascending=False)[["Sigla", "Estado", "2024"]].reset_index(drop=True)
rank_2024.index += 1   
rank_2024

## 4. Qual estado teve a maior melhora de IDH entre 1991 e 2024?

In [ ]:
melhora = df[["Sigla", "Estado", "1991", "2024"]].copy()   
melhora["melhora_1991_2024"] = (melhora["2024"] - melhora["1991"]).round(3)
melhora = melhora.sort_values("melhora_1991_2024", ascending=False)
print(melhora.head(5).to_string(index=False))

maior = melhora.iloc[0]
print(f"\nMaior melhora: {maior['Estado']} ({maior['Sigla']}), de {maior['1991']:.3f} para {maior['2024']:.3f} (+{maior['melhora_1991_2024']:.3f})")

## 5. Existe algum estado em que o IDH piorou?


In [ ]:
anos = [c for c in df.columns if str(c).isdigit()]


piorou = df[df["2024"] < df["1991"]]
print("Estados com IDH menor em 2024 do que em 1991:", piorou["Sigla"].tolist() or "nenhum")

In [ ]:

variacao = df.set_index("Sigla")[anos].diff(axis=1)

print("Quantidade de estados cujo IDH caiu em cada ano:")
print((variacao < 0).sum().loc[lambda s: s > 0].to_string())

print("\nMaiores quedas de um ano para o outro:")
quedas = variacao.stack().sort_values().head(5).round(3)
quedas

In [ ]:

df.loc[df["2024"] < df["2023"], ["Sigla", "Estado", "2023", "2024"]]

## 6. Passar do formato largo para o longo (`melt`)


In [ ]:
anos = [c for c in df.columns if str(c).isdigit()]
print(anos)

id_vars = [c for c in df.columns if c not in anos]
df_longo = df.melt(
    id_vars=id_vars,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH",
)
df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")
df_longo.head()

## 7. Plotar apenas Minas Gerais

In [1]:
mg = df_longo[df_longo["Sigla"] == "MG"].sort_values("Ano")

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(mg["Ano"], mg["IDH"], marker="o", linewidth=2, color="tab:blue")
for x, y in zip(mg["Ano"], mg["IDH"]):
    ax.annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.set_title("Evolução do IDH em Minas Gerais (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.4, 0.9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("idh_minas_gerais.png", dpi=150)
plt.show()

NameError: name 'df_longo' is not defined

## 8. Evolução do IDH de cada estado

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(grupo["Ano"], grupo["IDH"], marker="o", markersize=3, linewidth=1.5, label=sigla)

ax.set_title("Evolução do IDH por estado (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")
fig.tight_layout()
fig.savefig("idh_todos_estados.png", dpi=150)
plt.show()